# Multimodal Pipeline for RAG (Unstructured.io & Mistral)

## Phase 0: Setting up the Project

## Phase 1: Indexing

Indexing starts with the cleaning and extraction of raw data from formats like PDF, to converted it into structured formats.

### For Approach 1-2: Separate extracted elements into tables, text and images

#### Separate tables from texts

In [ ]:
# separate tables from texts
tables = []
texts = []

for chunk in chunks:
    if "CompositeElement" in str(type(chunk)):
        for el in chunk.metadata.orig_elements:
            if "Table" in str(type(el)):
                tables.append(el)
                #remove table from chunk
                chunk.metadata.orig_elements.remove(el)

In [ ]:
tables_html = [table.metadata.text_as_html for table in tables]
tables_html[1]

In [ ]:
# For Testing only!
# Search for Table in chunk
for chunk in chunks:
    if "Table" in str(type(chunk)):
        print("Found Table in chunk")
        print(chunk.metadata.orig_elements)
        #print(chunk.metadata

#### Get the images from the CompositeElement objects

In [ ]:
def get_images_base64(chunks):
    images_b64 = []
    for chunk in chunks:
        if "CompositeElement" in str(type(chunk)):
            chunk_els = chunk.metadata.orig_elements
            for el in chunk_els:
                if "Image" in str(type(el)):
                    images_b64.append(el.metadata.image_base64)
                    chunk.metadata.orig_elements.remove(el)
    return images_b64

images = get_images_base64(chunks)

In [ ]:
#Check if images are still in the chunks
for chunk in chunks:
    if "Image" in str(type(chunk)):
        print("Found Image in chunk")
        print(chunk.metadata.orig_elements)
        #print(chunk.metadata.orig_elements

#### The rest is text

In [ ]:
# The rest of the chunks are text
texts = []
for chunk in chunks:
    texts.append(chunk)

In [ ]:
#print(texts[0])

### 3. Approach: Mistral OCR API
-> Using Mistral OCR API for document processing and text extraction delivered the best results.

In [ ]:
%pip install mistralai

In [ ]:
from mistralai import Mistral
from pathlib import Path
from mistralai import DocumentURLChunk, ImageURLChunk, TextChunk
import json

api_key = os.environ.get("MISTRAL_API_KEY")
client = Mistral(api_key=api_key)

pdf_file = file_path

# Upload PDF file to Mistral's OCR service
uploaded_file = client.files.upload(
    file={
        "file_name": pdf_file,
        "content": open(pdf_file, "rb"),
    },
    purpose="ocr",
)

# Get URL for the uploaded file
signed_url = client.files.get_signed_url(file_id=uploaded_file.id, expiry=1)

# Process PDF with OCR, including embedded images
pdf_response = client.ocr.process(
    document=DocumentURLChunk(document_url=signed_url.url),
    model="mistral-ocr-latest",
    include_image_base64=True
)

# Convert response to JSON format
response_dict = json.loads(pdf_response.model_dump_json())

In [ ]:
response_dict

Separate markdown and images into lists

In [ ]:
markdown = []
images = []

for page in response_dict["pages"]:
    markdown.append(page["markdown"])
    if page["images"] is not None:
        for image in page["images"]:
            images.append(image["image_base64"])

Try to partition the markdown into text and tables with unstructured

In [ ]:
from unstructured.partition.md import partition_md

file_path = "test.md"

chunks = partition_md(
    filename=file_path,
)

In [ ]:
chunks

In [ ]:
tables = []

for chunk in chunks:
    if "Table" in str(type(chunk)):
        tables.append(chunk)
        chunks.remove(chunk)

In [ ]:
print(tables[1].text)

##### Test Function to recreate the pdf and check the results

In [ ]:
# with open('test.md', 'w') as f:
#         for text in markdown:
#             f.write(text)

In [ ]:
import base64

def data_uri_to_bytes(data_uri):
    _, encoded = data_uri.split(',', 1)
    return base64.b64decode(encoded)

def export_image(image):
    parsed_image = data_uri_to_bytes(image["image_base64"])
    with open(image["id"], 'wb') as file:
        file.write(parsed_image)

with open('output.md', 'w') as f: 
    for page in response_dict["pages"]:
        f.write(page["markdown"])
        for image in page["images"]:
            export_image(image)

In [ ]:
%pip install docling
%pip install ipywidgets

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions()
pipeline_options.do_picture_description = True
pipeline_options.generate_picture_images = True
pipeline_options.images_scale = 2
pipeline_options.do_picture_classification = True


converter = DocumentConverter(format_options={
    InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
})

source = file_path
result = converter.convert(source)

doc = result.document

In [ ]:
doc = result.document
print(doc.export_to_markdown())

#### Extract Tables

In [ ]:
tables = []

for table in doc.tables:
    tables.append(table.export_to_markdown(doc))

print(len(tables))

#### Extract the Images

In [ ]:
import base64

output_dir = Path("backend/assets/figures")
doc_filename = result.input.file.stem

# Create the output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

image_summaries = []
images = []

picture_counter = 0
for picture in doc.pictures:
    picture_counter += 1
    img = picture.get_image(doc)
    
    # Skips all Logo's
    if any(annotation.kind == "description" for annotation in picture.annotations):
        try:
            b64 = picture._image_to_base64(img)
            decoded_image = base64.b64decode(b64, validate=True)
            if decoded_image:
                images.append(b64)
        except Exception as e:
            print(f"Error decoding image: {e}")
        element_image_filename = (
            output_dir / f"{doc_filename}-picture-{picture_counter}.png")
        with element_image_filename.open("wb") as fp:
            picture.get_image(doc).save(fp, "PNG")
        for annotation in picture.annotations:
            if annotation.kind == "description":
                print(f"Image description: {annotation.text}")
                image_summaries.append(annotation.text)
    else:
        pass

In [ ]:
!brew install poppler tesseract libmagic
%pip install "unstructured[pdf, md]" pillow pdf2image

In [ ]:
%pip install --upgrade --quiet  langchain langchain-openai langchain-community
%pip install dotenv
%pip install pymongo

In [ ]:
%pip install -Uq chromadb tiktoken langchain-community langchain-chroma


In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.5,
    check_every_n_seconds=1,
    max_bucket_size=500000,
)
load_dotenv()
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")

## Phase 2: Chunking & Contextualization

Adding context to both images and tables as well as text for enhanced retrieval and understanding.<br>
Proposed by Anthropic "contextual-retrieval" approach.

### Contextualization of Chunks

In [ ]:
document_context_prompt = """
<document>
{doc_content}
</document>
"""

chunk_context_prompt = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""

messages = [
    (
        "user",
        [
            {"type": "text", "text": document_context_prompt.format(doc_content=doc.export_to_markdown())},
            {
                "type": "text",
                "text": chunk_context_prompt.format(chunk_content="{texts}"),
            },
        ],
    )
]

prompt = ChatPromptTemplate.from_messages(messages)

chain = prompt | model | StrOutputParser()

chunks_context = chain.batch(texts)

### Summarization of Tables

In [ ]:
# Prompt
prompt_text ="""
You are a helpful assistant tasked with summarizing tables precisely.
Give a concise summary of the table.

Respond only with the summary, no additional comment.
Do not start your message by saying "Here is a summary" or anything like that.
Just give the summary as it is.

Table: {element}

"""
#rate_limiter=rate_limiter,

# Summary chain
model = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    api_version="2024-12-01-preview",
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    temperature=0,
    model="gpt-4o"
)

prompt = ChatPromptTemplate.from_template(prompt_text)

#summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()
summarize_chain = prompt | model | StrOutputParser()

In [ ]:
# Only for Approach 1 & 2
# tables_html = [table.metadata.text_as_html for table in tables]
# tables_html[1]

In [ ]:
table_summaries = summarize_chain.batch(tables, {"max_concurrency": 3})

### Summarization of Images

In [ ]:
prompt_template = """You are a helpful assistant tasked with describing a image in detail. For context, the image is part of a design specification explaining the design of a digital temperature sensor.
Respond only with the description, no additional comment. Do not start your message by saying "Here is a description" or anything like that. Just give the description as it is."""
messages = [
    (
        "user",
        [
            {"type": "text", "text": prompt_template},
            {
                "type": "image_url",
                "image_url": {"url": "data:image/jpeg;base64,{image}"},
            },
        ],
    )
]

prompt = ChatPromptTemplate.from_messages(messages)

chain = prompt | model | StrOutputParser()

image_summaries = chain.batch(images)

In [ ]:
# Prompt
prompt_text ="""
You are a helpful assistant tasked with summarizing tables precisely.
Give a concise summary of the table.

Respond only with the summary, no additional comment.
Do not start your message by saying "Here is a summary" or anything like that.
Just give the summary as it is.

Table: {element}

"""
#rate_limiter=rate_limiter,

# Summary chain
model = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    api_version="2024-12-01-preview",
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    temperature=0,
    model="gpt-4o"
)

prompt = ChatPromptTemplate.from_template(prompt_text)

#summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()
summarize_chain = prompt | model | StrOutputParser()

In [ ]:
# Only for Approach 1 & 2
# tables_html = [table.metadata.text_as_html for table in tables]
# tables_html[1]

In [ ]:
table_summaries = summarize_chain.batch(tables, {"max_concurrency": 3})

## Phase 3: Vectorization

### Create Vectorstore

In [ ]:
!rm -rf backend/db

##### Local Store

In [ ]:
import uuid
from pathlib import Path
from langchain_chroma import Chroma
import chromadb
from langchain.storage import InMemoryStore
from langchain.schema.document import Document
from langchain_openai import AzureOpenAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever

azure_embeddings_endpoint = os.getenv("AZURE_OPENAI_EMBEDDINGS_ENDPOINT")

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    api_version="2024-12-01-preview",
    azure_endpoint=azure_embeddings_endpoint,
    api_key=azure_api_key,
)

vectorstore = Chroma(collection_name="DesignSpecsRAG",
                     embedding_function=embeddings,
                     )

# The storage layer for the parent documents
store = InMemoryStore()

id_key = "doc_id"
text_item = "text_item"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

#### Persistent Store

In [ ]:
import uuid
from pathlib import Path
from langchain_chroma import Chroma
import chromadb
from langchain.storage import InMemoryStore
from langchain.schema.document import Document
from langchain_openai import AzureOpenAIEmbeddings
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import LocalFileStore
import shutil

azure_embeddings_endpoint = os.getenv("AZURE_OPENAI_EMBEDDINGS_ENDPOINT")

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    api_version="2024-12-01-preview",
    azure_endpoint=azure_embeddings_endpoint,
    api_key=azure_api_key,
)

vectorstore_dir = Path("backend/db/chroma")
if vectorstore_dir.exists():
    shutil.rmtree(vectorstore_dir)

docs_dir = Path("backend/db/docs")
vectorstore_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(
    path=str(vectorstore_dir),
    settings=chromadb.Settings(
        allow_reset=True,  # Allows creation of new database
        is_persistent=True
    )
)

vectorstore = Chroma(collection_name="DesignSpecsRAG",
                     embedding_function=embeddings,
                     #persist_directory=vectorstore_dir
                     client=client
                     )

# The storage layer for the parent documents
store = LocalFileStore(root_path=str(docs_dir))

id_key = "doc_id"
text_item = "text_item"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

### Load Data

In [ ]:
# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=summary, metadata={id_key: doc_ids[i], text_item: texts[i]}) for i, summary in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=summary, metadata={id_key: table_ids[i]}) for i, summary in enumerate(table_summaries)
]
retriever.vectorstore.add_documents(summary_tables)
retriever.docstore.mset(list(zip(table_ids, tables)))

# Add image summaries
img_ids = [str(uuid.uuid4()) for _ in images]
summary_img = [
    Document(page_content=summary, metadata={id_key: img_ids[i]}) for i, summary in enumerate(image_summaries)
]
retriever.vectorstore.add_documents(summary_img)
retriever.docstore.mset(list(zip(img_ids, images)))

Note: Use open source Retriever like ChromaDB for open source!

#### Check Retriever

In [ ]:
# Retrieve
chunks = retriever.invoke(
    "What is the MTS2916A"
)

for chunk in chunks:
    print(str(doc) + "\n\n" + "-" * 80)